# 01 - Training (SFT, two from-scratch DPO runs, LoRA persona)

**Kaggle settings before running:** Accelerator **GPU T4 x2**, Internet **On**.

Recommended way to run:

1. Interactively run sections 1 to 5 (setup, tests, data) and the smoke cells of each stage to see that everything works.
2. Then use **Save Version -> Save & Run All (Commit)**. The commit runs the whole notebook in the background
   (no open browser tab needed) and keeps everything under `/kaggle/working` as the version's output.

Expected time on T4 x2: SFT about 1.5 h, each DPO run 2 to 2.5 h, persona 10 to 20 min (6.5 to 8.5 h in total,
inside the 12 h session limit). To split the work across sessions, set `STAGES` in section 6 and attach the earlier
version's output through **Add Input**: finished stages found there are skipped.

Everything is written under `/kaggle/working/small-model-alignment/outputs/<run_name>/`.

## 1. Get the code

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# ---- EDIT THESE ---------------------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # your GitHub repository
REPO_BRANCH = "main"
CODE_DATASET_DIR = ""   # alternative to GitHub: an uploaded copy, e.g. "/kaggle/input/my-pa2-code/genaiPA2"
# -------------------------------------------------------------------------------------------

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "small-model-alignment"

def find_code_in_inputs():
    """A copy of the repository inside an attached input (an uploaded dataset or an earlier notebook's output)."""
    for marker in sorted(Path("/kaggle/input").glob("**/scripts/judge_pairwise_kaggle.py")):
        return marker.parents[1]
    return None

if REPO_DIR.exists():
    print("repository already present:", REPO_DIR)
else:
    source = Path(CODE_DATASET_DIR) if CODE_DATASET_DIR else None
    if source is None and "YOUR_USERNAME" in REPO_URL:
        source = find_code_in_inputs()
        if source is None:
            raise SystemExit("Set REPO_URL to your GitHub repository, or set CODE_DATASET_DIR, or attach an input "
                             "that contains the code.")
    if source is not None:
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("outputs", ".git", ".venv", "__pycache__"))
        print("copied code from", source)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"   # show training / generation logs live in "!python ..." cells
sys.path.insert(0, str(REPO_DIR))
print("working directory:", os.getcwd())

## 2. Install dependencies (PyTorch is left untouched)

In [ ]:
# Keeps Kaggle's CUDA build of PyTorch: torch is never installed or pinned here.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.56.2,<5" "datasets>=4.7,<5" huggingface_hub "accelerate>=1.4.0" tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.17.0" "trl==1.14.1" pytest
!python -m pip install -q -e . --no-deps

## 3. Check the environment: two T4 GPUs and package versions

In [ ]:
!nvidia-smi
!python scripts/check_env.py --require_gpus 2

## 4. Offline tests (adapter contract and training math; no downloads)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py
!PYTHONPATH=. python -m pytest -q tests

## 5. Choose stages and look for finished runs

`finished(run)` finds a completed run in this session or in any attached input, so a later session never repeats
a stage that already finished.

In [ ]:
STAGES = ["sft", "dpo_hh", "dpo_ultrafeedback", "persona_lora"]   # remove entries to run only part of the pipeline
RUN_SMOKE = True                                                  # 4-step check before each full run

In [ ]:
from pathlib import Path

def finished(run_name):
    """Directory of a finished run: outputs/<run_name> in this session, else inside any attached input."""
    local = Path("outputs") / run_name
    if (local / "TRAINING_COMPLETE.json").exists():
        return local
    matches = sorted(Path("/kaggle/input").glob(f"**/outputs/{run_name}/TRAINING_COMPLETE.json"))
    return matches[-1].parent if matches else None

for name in ("sft", "dpo_hh", "dpo_ultrafeedback", "persona_lora"):
    print(f"{name:18s} {finished(name) or 'not finished yet'}")

## 6. Prepare data

Builds the single-turn files in `outputs/data/` (30k/1k SFT, 15k/1k HH, 15k/1k UltraFeedback) from pinned dataset
revisions and writes a `*_provenance.json` next to each. Prompts of the fixed evaluation sets are excluded.
Existing files are kept.

In [ ]:
!python scripts/make_alpaca_eval_subset.py --verify
if "sft" in STAGES and finished("sft") is None:
    !python scripts/prepare_data.py --task sft
if "dpo_hh" in STAGES and finished("dpo_hh") is None:
    !python scripts/prepare_data.py --task hh
if "dpo_ultrafeedback" in STAGES and finished("dpo_ultrafeedback") is None:
    !python scripts/prepare_data.py --task ultrafeedback
!ls -la outputs/data

## 7. SFT

A 4-step smoke run first (its checkpoints are deleted automatically), then the real run. The resume state goes to
`/kaggle/temp`, which does not count against the 20 GB output quota and survives a kernel restart in the same session.

In [ ]:
if "sft" in STAGES and finished("sft") is None:
    if RUN_SMOKE:
        !python scripts/train_sft.py --config configs/sft.yaml --smoke
    !python scripts/train_sft.py --config configs/sft.yaml --resume --override resume_every_steps=250 resume_dir=/kaggle/temp/resume/sft
    !python scripts/package_outputs.py --source outputs/sft outputs/data --zip_path /kaggle/working/sft_run_outputs.zip
SFT_BEST = finished("sft") / "best" if finished("sft") else None
print("SFT best checkpoint:", SFT_BEST)

## 8. DPO on HH-RLHF

Policy on `cuda:0`, frozen reference on `cuda:1`, both loaded from the SFT best checkpoint.
The step-0 validation must show reward accuracy 0.0 and margin 0.0: policy and reference are identical, so all
rewards tie and the strict comparison counts none as correct.

In [ ]:
if "dpo_hh" in STAGES and finished("dpo_hh") is None:
    assert SFT_BEST is not None, "SFT must be finished first: run it above or attach its output with Add Input"
    if RUN_SMOKE:
        !python scripts/train_dpo.py --config configs/dpo_hh.yaml --smoke --override sft_checkpoint=$SFT_BEST
    !python scripts/train_dpo.py --config configs/dpo_hh.yaml --resume --override sft_checkpoint=$SFT_BEST resume_every_steps=100 resume_dir=/kaggle/temp/resume/dpo_hh
    !python scripts/package_outputs.py --source outputs/dpo_hh --zip_path /kaggle/working/dpo_hh_run_outputs.zip

## 9. DPO on UltraFeedback (same code, same SFT checkpoint, different data)

In [ ]:
if "dpo_ultrafeedback" in STAGES and finished("dpo_ultrafeedback") is None:
    assert SFT_BEST is not None, "SFT must be finished first: run it above or attach its output with Add Input"
    !python scripts/train_dpo.py --config configs/dpo_ultrafeedback.yaml --resume --override sft_checkpoint=$SFT_BEST resume_every_steps=100 resume_dir=/kaggle/temp/resume/dpo_ultrafeedback
    !python scripts/package_outputs.py --source outputs/dpo_ultrafeedback --zip_path /kaggle/working/dpo_ultrafeedback_run_outputs.zip

## 10. Persona: TRL DPOTrainer + LoRA on SmolLM2-360M-Instruct

In [ ]:
if "persona_lora" in STAGES and finished("persona_lora") is None:
    if RUN_SMOKE:
        !python scripts/train_persona_lora.py --config configs/persona_lora.yaml --smoke
    !python scripts/train_persona_lora.py --config configs/persona_lora.yaml
    !python scripts/package_outputs.py --source outputs/persona_lora --zip_path /kaggle/working/persona_lora_run_outputs.zip --include_weights

## 11. Training plots and summary tables

In [ ]:
RUN_DIRS = " ".join(str(finished(name)) for name in ("sft", "dpo_hh", "dpo_ultrafeedback", "persona_lora") if finished(name))
if RUN_DIRS:
    !python scripts/plot_metrics.py --run_dir $RUN_DIRS --out_dir outputs/plots
    !python scripts/summarize_results.py --runs $RUN_DIRS --out_dir outputs/report
    !python scripts/package_outputs.py --source outputs/plots outputs/report --zip_path /kaggle/working/training_plots_and_report.zip

In [ ]:
from IPython.display import Image, display
for path in sorted(Path("outputs/plots").glob("*_curves.png")):
    print(path)
    display(Image(filename=str(path)))

## 12. Disk use and saving

The three Qwen runs hold about 12 GB of checkpoints (best + final, fp32); the output limit is 20 GB.
The zip files contain metrics, configs, logs and plots only (no weights), so they are small and easy to download.

- If you ran this through **Save & Run All (Commit)**, the outputs are already attached to the saved version.
- If you ran it interactively, use **Save Version -> Quick Save** and enable *Always save output when creating a
  Quick Save* under Advanced Settings.

Then open notebook 02 and attach this notebook with **Add Input -> Notebooks**.

**Restarting:** run all cells again. A finished stage is skipped. An interrupted stage continues from
`/kaggle/temp/resume/<run>` while the session is alive and restarts from its beginning in a new session.

In [ ]:
!du -sh outputs/* /kaggle/working/*.zip 2>/dev/null
!df -h /kaggle/working | tail -1